In [2]:
# =============================================================================
# SATD CLASSIFICATION PROJECT — FULL PIPELINE OVERVIEW
# =============================================================================
# Goal:
#   Test whether two modern text embedding models (Qwen3 and E5-Large) can
#   classify "Self-Admitted Technical Debt" (SATD) comments better than the
#   method used in the original replication study (GloVe+BiLSTM / BERT).
#
# We compare results across 4 artifact types:
#   - Code Comments
#   - Issues
#   - Commit Messages
#   - Pull Requests
#
# Label scheme (Option A — matches the replication study exactly):
#   - Not-SATD  : not technical debt
#   - C/D       : code or design debt (merged from CODE, DESIGN, CODE/DESIGN)
#   - REQ       : requirement debt
#   - TES       : test debt
#   - DOC       : documentation debt
#   (Defect, Architecture, Build debt are dropped — not used in replication)
# =============================================================================


# -----------------------------------------------------------------------------
# PHASE 1 — CLEAN THE DATA  ✓ DONE
# -----------------------------------------------------------------------------
# What we did:
#   1. Loaded all 31 original source files and tagged each row with its
#      artifact type (code_comment / issue / commit / pull_request) based
#      on the filename.
#   2. Standardised column names across files (different files used different
#      names for the text and label columns).
#   3. Dropped rows labeled as Defect, Architecture, or Build debt — these
#      were not used in the replication study we are comparing against.
#   4. Merged CODE, DESIGN, and CODE/DESIGN labels into one "C/D" bucket.
#   5. Dropped rows where the text is blank or too short (2 words or fewer).
#   6. Removed duplicate rows (same text + same artifact type).
#   7. Saved the cleaned file as the single starting point for all later steps.
#
# Final row counts after cleaning:
#   code_comment   353,009 rows  (C/D: 11,600 | REQ: 5,406 | TES: 1,786 | DOC: 1,410 | Not-SATD: 332,753)
#   issue           23,128 rows  (C/D:  2,189 | REQ:   814 | TES:   965 | DOC: 1,081 | Not-SATD:  18,079)
#   commit           5,616 rows  (C/D:    487 | REQ:   390 | TES:   372 | DOC:   392 | Not-SATD:   3,975)
#   pull_request     4,893 rows  (C/D:    502 | REQ:   155 | TES:   247 | DOC:   380 | Not-SATD:   3,609)
#
# Output file: satd_with_artifact_type.csv
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PHASE 2 — BALANCE THE DATA
# -----------------------------------------------------------------------------
# Why we need this:
#   Some categories have far more examples than others. For example, Not-SATD
#   has hundreds of thousands of rows while some SATD subtypes have only ~155.
#   A model trained on imbalanced data just learns to always predict Not-SATD.
#
# How we fix it (per artifact type):
#   - Find the largest SATD class → use its size as the TARGET for all classes.
#   - Not-SATD      : randomly downsample to the target size.
#   - SATD classes below target : use T5-based paraphrase model
#     (humarin/chatgpt_paraphraser_on_T5_base) running on Kaggle GPU to
#     generate reworded versions of existing examples until each class
#     reaches the target size.
#   - Special case — pull_request REQ (only 155 examples): cap augmentation
#     at 3x the original count (~465) to avoid repetitive low-quality text.
#
# Note: Qwen3 is reserved for Phase 4 (embeddings) where it is more suitable.
#
# Output file: satd_balanced.csv
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PHASE 3 — SPLIT THE DATA
# -----------------------------------------------------------------------------
# We divide each artifact type into three groups:
#   - 80% Training set   : used to teach the model
#   - 10% Validation set : used to tune the model during development
#   - 10% Test set       : used only at the end to measure real performance
#
# Important: we use the same split ratio as the original replication study
# so our results are directly comparable to their numbers.
# The exact split is saved so every model we test uses identical test data.
#
# Output files: satd_train.csv, satd_val.csv, satd_test.csv
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PHASE 4 — CONVERT TEXT TO NUMBERS (EMBEDDINGS)
# -----------------------------------------------------------------------------
# Machine learning models cannot read raw text — they need numbers.
# We run all text through two different embedding models, both run
# entirely on Kaggle (GPU T4 x2, no local computation needed):
#
#   - Qwen3-Embedding   : a newer multilingual embedding model
#                         (e.g. Qwen/Qwen3-Embedding-0.6B from Hugging Face)
#   - E5-Large          : a strong general-purpose embedding model
#                         (intfloat/e5-large-v2 from Hugging Face)
#
# How it works technically:
#   1. Load satd_train.csv, satd_val.csv, satd_test.csv (from Phase 3).
#   2. Load each embedding model + tokenizer via `transformers`.
#   3. Process text in batches (batch_size ~32-64) to fit in GPU memory.
#   4. For E5 models: prefix text with "query: " or "passage: " as required
#      by the model's training format (E5 is prefix-sensitive).
#   5. Mean-pool or use the model's pooling layer to get one fixed-length
#      vector per text (commonly 768-1024 dimensions depending on model).
#   6. L2-normalize embeddings if required by the model card.
#   7. Stack all vectors into a single NumPy array per split, aligned with
#      the row order of the corresponding CSV (so labels stay matched).
#   8. Save each array as .npy for fast loading in Phase 5.
#
# This step is repeated independently for each artifact type, OR all
# artifact types can be embedded together and filtered later — we will
# embed all text in one pass and keep the artifact_type column for
# filtering during training (simpler, avoids redundant model loading).
#
# HF_TOKEN should be set as a Kaggle secret before this phase to avoid
# rate limits when downloading the larger embedding models.
#
# Output files (saved to /kaggle/working/):
#   qwen3_train.npy   qwen3_val.npy   qwen3_test.npy
#   e5_train.npy      e5_val.npy      e5_test.npy
#   (plus train_labels.csv / val_labels.csv / test_labels.csv carried over
#    from Phase 3 — embeddings and labels are matched by row order)
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PHASE 5 — TRAIN THE CLASSIFIERS
# -----------------------------------------------------------------------------
# We train two simple classifier models on top of each set of embeddings,
# all done on Kaggle (CPU is sufficient for this phase — these are
# lightweight classifiers compared to the embedding models):
#
#   - XGBoost             : a tree-based gradient boosting model, good with
#                            structured/numeric input like embeddings
#   - Logistic Regression : a simple, fast linear baseline classifier
#
# This gives us 4 combinations in total:
#   1. Qwen3    embeddings + XGBoost
#   2. Qwen3    embeddings + Logistic Regression
#   3. E5-Large embeddings + XGBoost
#   4. E5-Large embeddings + Logistic Regression
#
# How it works technically:
#   1. Load the .npy embedding arrays + matching label CSVs from Phase 4.
#   2. Filter by artifact_type (code_comment / issue / commit / pull_request)
#      so each model is trained separately per artifact type.
#   3. Fit XGBoost (multi-class objective, e.g. "multi:softprob") and
#      Logistic Regression (multi_class="multinomial" or one-vs-rest)
#      on the training embeddings + labels.
#   4. Use the validation set to tune key hyperparameters:
#        - XGBoost: max_depth, n_estimators, learning_rate
#        - LogisticRegression: C (regularization strength)
#   5. Save each trained model (e.g. via joblib/pickle) for reuse in Phase 6.
#
# Total models trained = 4 combinations × 4 artifact types = 16 models.
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# PHASE 6 — COMPARE RESULTS
# -----------------------------------------------------------------------------
# We evaluate all 4 combinations on the test set and compare them against
# the original replication study's numbers (GloVe+BiLSTM and BERT).
#
# Metrics used (same as the replication study):
#   - Precision, Recall, F1-score per class
#   - Macro-averaged F1 overall
#
# Results are broken down by artifact type so we can see where each
# embedding model performs better or worse than the original method.
# -----------------------------------------------------------------------------

In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/shahriarpias/combined-text-label/satd_combined.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/argouml_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/jruby_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/cppsatd_cleaned_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/ownership argouml_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/SSATD_COMMITS_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/data-augmentation-pull-requests_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/manual_annotations_cleaned_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/data-augmentation-commit-messages_cleaned.csv
/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources/mlsatd_cleaned.csv
/kaggle/input/datasets/sha

In [4]:
# -----------------------------------------------------------------------------
# PHASE 1 — CLEAn THE DATA
# -----------------------------------------------------------------------------

In [5]:
import pandas as pd

# Load the combined dataset
df = pd.read_csv("/kaggle/input/datasets/shahriarpias/combined-text-label/satd_combined.csv")

print("Total rows:", len(df))
print("Unique labels:", df['label'].nunique())

# 1. Full label distribution (not just top 25)
label_counts = df['label'].value_counts()
label_counts.to_csv("full_label_distribution.csv")
print("\nSaved full label distribution to full_label_distribution.csv")
print(label_counts.head(50))

# 2. Check for duplicate text entries across the merged files
dupes = df[df.duplicated(subset=['text'], keep=False)]
print("\nRows with duplicate text:", len(dupes))
print("Unique duplicate text values:", dupes['text'].nunique())

# 3. Check for junk/short entries
df['word_count'] = df['text'].astype(str).str.split().str.len()
print("\nWord count summary:")
print(df['word_count'].describe())
print("Rows with <=2 words:", (df['word_count'] <= 2).sum())
print("Blank/null text rows:", df['text'].isna().sum())

Total rows: 820885
Unique labels: 1637

Saved full label distribution to full_label_distribution.csv
label
NON-SATD                                               537998
non_debt                                               161235
WITHOUT_CLASSIFICATION                                  54204
DESIGN/CODE                                             16402
REQUIREMENT                                              7698
DESIGN                                                   4885
documentation_debt                                       4661
code_debt                                                4553
code/design_debt                                         4520
requirement_debt                                         4041
test_debt                                                3432
design_debt                                              2660
DEFECT                                                   2338
CODE                                                     1965
TEST                     

In [6]:
import pandas as pd
import re

# Update this path to match your Kaggle input location
df = pd.read_csv("/kaggle/input/datasets/shahriarpias/combined-text-label/satd_combined.csv")

# 1. Drop exact duplicate text rows (keep first occurrence)
before = len(df)
df = df.drop_duplicates(subset=['text'], keep='first')
print(f"Dropped {before - len(df)} duplicate rows -> {len(df)} remain")

# 2. Drop short/junk text (<=2 words)
df['word_count'] = df['text'].astype(str).str.split().str.len()
before = len(df)
df = df[df['word_count'] > 2]
print(f"Dropped {before - len(df)} short/junk text rows -> {len(df)} remain")

# 3. Normalize label strings so case/punctuation differences don't matter
def normalize(label):
    label = str(label).strip().lower()
    label = re.sub(r'[\s_/\-]+', '_', label)
    return label

df['label_norm'] = df['label'].apply(normalize)

# 4. Define which normalized labels map to which schema bucket
NOT_SATD = {'non_satd', 'non_debt'}
UNCATEGORIZED_SATD = {'without_classification'}
CD = {'design_code', 'code_design', 'design_debt', 'code_debt', 'design', 'code'}
REQ = {'requirement', 'requirement_debt', 'requirements'}
DOC = {'documentation', 'documentation_debt'}
TES = {'test', 'test_debt'}

def map_label(norm):
    if norm in NOT_SATD:
        return 'Not-SATD', None
    if norm in UNCATEGORIZED_SATD:
        return 'SATD', None
    if norm in CD:
        return 'SATD', 'C/D'
    if norm in REQ:
        return 'SATD', 'REQ'
    if norm in DOC:
        return 'SATD', 'DOC'
    if norm in TES:
        return 'SATD', 'TES'
    return None, None  # everything else gets dropped

mapped = df['label_norm'].apply(lambda x: pd.Series(map_label(x)))
mapped.columns = ['id_label', 'cat_label']
df = pd.concat([df, mapped], axis=1)

before = len(df)
df = df.dropna(subset=['id_label'])
print(f"Dropped {before - len(df)} rows with unmapped/unrecognized labels -> {len(df)} remain")

# 5. Save the cleaned dataset
df_out = df[['text', 'id_label', 'cat_label']]
df_out.to_csv("satd_cleaned.csv", index=False)

print("\nFinal identification label counts:")
print(df_out['id_label'].value_counts())
print("\nFinal categorization label counts (SATD rows only):")
print(df_out['cat_label'].value_counts(dropna=True))

Dropped 388219 duplicate rows -> 432666 remain
Dropped 41641 short/junk text rows -> 391025 remain
Dropped 5569 rows with unmapped/unrecognized labels -> 385456 remain

Final identification label counts:
id_label
Not-SATD    349515
SATD         35941
Name: count, dtype: int64

Final categorization label counts (SATD rows only):
cat_label
C/D    13943
REQ     6420
TES     3372
DOC     3261
Name: count, dtype: int64


In [7]:
import pandas as pd
import os

INPUT_DIR = "/kaggle/input/datasets/shahriarpias/satd-sources-cleaned/satd sources"
OUTPUT_PATH = "/kaggle/working/satd_with_artifact_type.csv"

file_artifact_map = {
    # code comments
    "apache_ant_cleaned.csv":                       "code_comment",
    "apache_jmeter_cleaned.csv":                    "code_comment",
    "argouml_cleaned.csv":                          "code_comment",
    "jfreechart_cleaned.csv":                       "code_comment",
    "jruby_cleaned.csv":                            "code_comment",
    "cppsatd_cleaned_cleaned.csv":                  "code_comment",
    "mlsatd_cleaned.csv":                           "code_comment",
    "manual_annotations_cleaned_cleaned.csv":       "code_comment",
    "labeled_dataset_cleaned.csv":                  "code_comment",
    "OBrien_789_v2_cleaned.csv":                    "code_comment",
    "satd-dataset-code_comments_cleaned.csv":       "code_comment",
    "data-augmentation-code_comments_cleaned.csv":  "code_comment",
    "maldonado_corrected_cleaned.csv":              "code_comment",
    "duplicate_satd_comment_cleaned.csv":           "code_comment",
    "unique_satd_comment_cleaned.csv":              "code_comment",
    "SSATD_COMMENTS_cleaned.csv":                   "code_comment",
    "satd-comments-manual-subclass_cleaned.csv":    "code_comment",
    "ownership argouml_cleaned.csv":                "code_comment",
    "ownership_ant_cleaned.csv":                    "code_comment",
    "ownership_jmeter_cleaned.csv":                 "code_comment",
    "ownership_jruby_cleaned.csv":                  "code_comment",
    # issues
    "issue-satd_cleaned.csv":                       "issue",
    "satd-dataset-issues_cleaned.csv":              "issue",
    "data-augmentation-issues_cleaned.csv":         "issue",
    "SSATD_ISSUES_cleaned.csv":                     "issue",
    # commits
    "satd-dataset-commit_messages_cleaned.csv":     "commit",
    "data-augmentation-commit-messages_cleaned.csv":"commit",
    "SSATD_COMMITS_cleaned.csv":                    "commit",
    # pull requests
    "satd-dataset-pull_requests_cleaned.csv":       "pull_request",
    "data-augmentation-pull-requests_cleaned.csv":  "pull_request",
    "SSATD_PULL_cleaned.csv":                       "pull_request",
}

label_map = {
    "non-satd": "Not-SATD", "non_debt": "Not-SATD",
    "design/code": "C/D", "code/design_debt": "C/D", "design_debt": "C/D",
    "code_debt": "C/D", "design": "C/D", "code": "C/D",
    "design/code_debt": "C/D", "code/design": "C/D",
    "requirement": "REQ", "requirement_debt": "REQ", "requirements": "REQ",
    "documentation": "DOC", "documentation_debt": "DOC",
    "test": "TES", "test_debt": "TES",
    "without_classification": "SATD-uncat",
    "defect": None, "defect_debt": None,
    "architecture": None, "architecture_debt": None,
    "build": None, "build_debt": None,
    "implementation": None,
}

TEXT_COLS  = ["satd", "commenttext", "comment text", "text"]
LABEL_COLS = ["classification", "annotation", "manual_annotation",
              "software td type", "label", "debt"]

frames = []

for fname, artifact_type in file_artifact_map.items():
    fpath = os.path.join(INPUT_DIR, fname)
    if not os.path.exists(fpath):
        print(f"MISSING: {fname}")
        continue

    try:
        df = pd.read_csv(fpath, low_memory=False)
    except Exception as e:
        print(f"ERROR reading {fname}: {e}")
        continue

    df.columns = df.columns.str.strip().str.lower()

    text_col  = next((c for c in TEXT_COLS  if c in df.columns), None)
    label_col = next((c for c in LABEL_COLS if c in df.columns), None)

    if not text_col or not label_col:
        print(f"SKIPPED (cols not found) {fname}: {list(df.columns)}")
        continue

    out = pd.DataFrame({
        "text":          df[text_col].astype(str),
        "raw_label":     df[label_col].astype(str),
        "artifact_type": artifact_type,
        "source_file":   fname,
    })
    frames.append(out)
    print(f"Loaded {len(out):>7,} rows  [{artifact_type}]  {fname}")

combined = pd.concat(frames, ignore_index=True)
print(f"\nTotal before cleaning: {len(combined):,}")

# map labels
combined["mapped_label"] = combined["raw_label"].str.strip().str.lower().map(label_map)

# drop unwanted labels
before = len(combined)
combined = combined[combined["mapped_label"].notna()]
print(f"Dropped {before - len(combined):,} unrecognised/unwanted labels")

# drop short/junk text
combined["word_count"] = combined["text"].str.split().str.len()
before = len(combined)
combined = combined[combined["word_count"] > 2]
print(f"Dropped {before - len(combined):,} short/junk text rows")

# drop duplicates
before = len(combined)
combined = combined.drop_duplicates(subset=["text", "artifact_type"])
print(f"Dropped {before - len(combined):,} duplicate rows")

print("\nRows by artifact type:")
print(combined["artifact_type"].value_counts())

print("\nLabel distribution per artifact type:")
print(combined.groupby(["artifact_type","mapped_label"]).size().unstack(fill_value=0))

combined[["text","mapped_label","artifact_type","source_file"]].to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Loaded     116 rows  [code_comment]  apache_ant_cleaned.csv
Loaded     277 rows  [code_comment]  apache_jmeter_cleaned.csv
Loaded   1,005 rows  [code_comment]  argouml_cleaned.csv
Loaded     206 rows  [code_comment]  jfreechart_cleaned.csv
Loaded     219 rows  [code_comment]  jruby_cleaned.csv
Loaded 529,013 rows  [code_comment]  cppsatd_cleaned_cleaned.csv
Loaded     760 rows  [code_comment]  mlsatd_cleaned.csv
Loaded  34,891 rows  [code_comment]  manual_annotations_cleaned_cleaned.csv
Loaded   1,036 rows  [code_comment]  labeled_dataset_cleaned.csv
Loaded     760 rows  [code_comment]  OBrien_789_v2_cleaned.csv
Loaded  58,273 rows  [code_comment]  satd-dataset-code_comments_cleaned.csv
Loaded  47,956 rows  [code_comment]  data-augmentation-code_comments_cleaned.csv
Loaded  58,273 rows  [code_comment]  maldonado_corrected_cleaned.csv
Loaded     602 rows  [code_comment]  duplicate_satd_comment_cleaned.csv
Loaded     512 rows  [code_comment]  unique_satd_comment_cleaned.csv
Loaded     17

In [8]:
# -----------------------------------------------------------------------------
# PHASE 2 — BALANCE THE DATA
# -----------------------------------------------------------------------------

In [9]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

INPUT_PATH  = "/kaggle/working/satd_with_artifact_type.csv"
OUTPUT_PATH = "/kaggle/working/satd_balanced.csv"

# ── LOAD MODEL ────────────────────────────────────────────────────────────────
print("Loading paraphrase model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base")
model = AutoModelForSeq2SeqLM.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base").to(device)
print("Model loaded.")

# ── PARAPHRASE FUNCTION ───────────────────────────────────────────────────────
def paraphrase_batch(texts, n_per_text=5):
    """Generate n paraphrases for each text in the batch."""
    results = []
    for text in texts:
        text = str(text)[:256]  # cap length to avoid slowdowns
        input_ids = tokenizer(
            f"paraphrase: {text}",
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).input_ids.to(device)

        outputs = model.generate(
            input_ids,
            num_return_sequences=n_per_text,
            num_beams=max(n_per_text, 5),
            max_length=128,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
        for out in outputs:
            decoded = tokenizer.decode(out, skip_special_tokens=True).strip()
            if decoded and decoded.lower() != text.lower():
                results.append(decoded)
    return results

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH)
df_cat = df[df["mapped_label"] != "SATD-uncat"].copy()

def get_target(group):
    satd_only = group[group["mapped_label"] != "Not-SATD"]
    return int(satd_only["mapped_label"].value_counts().max())

# ── BALANCE ───────────────────────────────────────────────────────────────────
balanced_frames = []

for artifact_type, group in df_cat.groupby("artifact_type"):
    target = get_target(group)
    print(f"\n── {artifact_type}  (target per class: {target:,}) ──")

    artifact_frames = []

    for label, subset in group.groupby("mapped_label"):
        current = len(subset)

        if label == "Not-SATD":
            sampled = subset.sample(n=target, random_state=42)
            artifact_frames.append(sampled)
            print(f"  {label}: {current:,} → downsampled to {target:,}")

        elif current >= target:
            sampled = subset.sample(n=target, random_state=42)
            artifact_frames.append(sampled)
            print(f"  {label}: {current:,} → sampled to {target:,}")

        else:
            # special cap for pull_request REQ
            if artifact_type == "pull_request" and label == "REQ":
                needed = min(target - current, current * 3)
                print(f"  {label}: {current:,} → augmenting +{needed} (capped at 3x)")
            else:
                needed = target - current
                print(f"  {label}: {current:,} → augmenting +{needed}")

            artifact_frames.append(subset)

            # generate paraphrases in batches
            new_rows = []
            texts = subset["text"].tolist()
            batch_size = 16
            idx = 0

            while len(new_rows) < needed:
                batch_texts = [texts[i % len(texts)]
                               for i in range(idx, idx + batch_size)]
                paraphrases = paraphrase_batch(batch_texts, n_per_text=5)

                for p in paraphrases:
                    if len(new_rows) >= needed:
                        break
                    new_rows.append({
                        "text":          p,
                        "mapped_label":  label,
                        "artifact_type": artifact_type,
                        "source_file":   "augmented"
                    })

                idx += batch_size
                if (idx // batch_size) % 10 == 0:
                    print(f"    ... {len(new_rows)}/{needed} generated")

            aug_df = pd.DataFrame(new_rows[:needed])
            artifact_frames.append(aug_df)

    balanced_frames.append(pd.concat(artifact_frames, ignore_index=True))

# ── SAVE ──────────────────────────────────────────────────────────────────────
balanced = pd.concat(balanced_frames, ignore_index=True)

print("\n\nFinal balanced distribution:")
print(balanced.groupby(["artifact_type", "mapped_label"]).size().unstack(fill_value=0))

balanced.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Loading paraphrase model...
Using device: cuda


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded.

── code_comment  (target per class: 11,600) ──
  C/D: 11,600 → sampled to 11,600
  DOC: 1,410 → augmenting +10190
    ... 800/10190 generated
    ... 1600/10190 generated
    ... 2400/10190 generated
    ... 3200/10190 generated
    ... 4000/10190 generated
    ... 4800/10190 generated
    ... 5600/10190 generated
    ... 6400/10190 generated
    ... 7200/10190 generated
    ... 8000/10190 generated
    ... 8800/10190 generated
    ... 9600/10190 generated
  Not-SATD: 332,753 → downsampled to 11,600
  REQ: 5,406 → augmenting +6194
    ... 800/6194 generated
    ... 1598/6194 generated
    ... 2398/6194 generated
    ... 3197/6194 generated
    ... 3997/6194 generated
    ... 4797/6194 generated
    ... 5597/6194 generated
  TES: 1,786 → augmenting +9814
    ... 800/9814 generated
    ... 1600/9814 generated
    ... 2400/9814 generated
    ... 3200/9814 generated
    ... 4000/9814 generated
    ... 4800/9814 generated
    ... 5600/9814 generated
    ... 6400/9814 generate

In [ ]:
# -----------------------------------------------------------------------------
# PHASE 3 — SPLIT THE DATA
# -----------------------------------------------------------------------------

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

# If continuing in the same session, 'balanced' already exists in memory.
# If running fresh, uncomment the line below:
# balanced = pd.read_csv("/kaggle/working/satd_balanced.csv")

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
RANDOM_SEED = 42

train_frames, val_frames, test_frames = [], [], []

print("Splitting per artifact_type + label (stratified)...\n")

for (artifact_type, label), group in balanced.groupby(["artifact_type", "mapped_label"]):
    # first split off train (80%) vs temp (20%)
    train_part, temp_part = train_test_split(
        group, train_size=TRAIN_RATIO, random_state=RANDOM_SEED
    )
    # split temp (20%) into val (10%) and test (10%) → 50/50 of temp
    val_part, test_part = train_test_split(
        temp_part, train_size=0.5, random_state=RANDOM_SEED
    )

    train_frames.append(train_part)
    val_frames.append(val_part)
    test_frames.append(test_part)

    print(f"  {artifact_type:13s} | {label:9s} → train: {len(train_part):5,}  val: {len(val_part):5,}  test: {len(test_part):5,}")

train_df = pd.concat(train_frames, ignore_index=True)
val_df   = pd.concat(val_frames, ignore_index=True)
test_df  = pd.concat(test_frames, ignore_index=True)

print(f"\nTotal rows → train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

# sanity check: ratios should be ~80/10/10
total = len(train_df) + len(val_df) + len(test_df)
print(f"\nRatios → train: {len(train_df)/total:.1%}  val: {len(val_df)/total:.1%}  test: {len(test_df)/total:.1%}")

# add a split_id column for traceability (so we always know which row went where)
train_df["split"] = "train"
val_df["split"]   = "val"
test_df["split"]  = "test"

# save each split separately
train_df.to_csv("/kaggle/working/satd_train.csv", index=False)
val_df.to_csv("/kaggle/working/satd_val.csv", index=False)
test_df.to_csv("/kaggle/working/satd_test.csv", index=False)

print("\nSaved: satd_train.csv, satd_val.csv, satd_test.csv")

Splitting per artifact_type + label (stratified)...

  code_comment  | C/D       → train: 9,280  val: 1,160  test: 1,160
  code_comment  | DOC       → train: 9,280  val: 1,160  test: 1,160
  code_comment  | Not-SATD  → train: 9,280  val: 1,160  test: 1,160
  code_comment  | REQ       → train: 9,280  val: 1,160  test: 1,160
  code_comment  | TES       → train: 9,280  val: 1,160  test: 1,160
  commit        | C/D       → train:   389  val:    49  test:    49
  commit        | DOC       → train:   389  val:    49  test:    49
  commit        | Not-SATD  → train:   389  val:    49  test:    49
  commit        | REQ       → train:   389  val:    49  test:    49
  commit        | TES       → train:   389  val:    49  test:    49
  issue         | C/D       → train: 1,751  val:   219  test:   219
  issue         | DOC       → train: 1,751  val:   219  test:   219
  issue         | Not-SATD  → train: 1,751  val:   219  test:   219
  issue         | REQ       → train: 1,751  val:   219  test:  

In [ ]:
# -----------------------------------------------------------------------------
# PHASE 4 — CONVERT TEXT TO NUMBERS (EMBEDDINGS)
# -----------------------------------------------------------------------------

In [11]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from kaggle_secrets import UserSecretsClient

# ── LOAD HF TOKEN FROM KAGGLE SECRETS ────────────────────────────────────────
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── DATA (already in memory from Phase 3; reload if needed) ─────────────────
# train_df = pd.read_csv("/kaggle/working/satd_train.csv")
# val_df   = pd.read_csv("/kaggle/working/satd_val.csv")
# test_df  = pd.read_csv("/kaggle/working/satd_test.csv")

splits = {"train": train_df, "val": val_df, "test": test_df}

# ── MEAN POOLING HELPER ──────────────────────────────────────────────────────
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch, seq_len, hidden)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

# ── GENERIC EMBEDDING FUNCTION ───────────────────────────────────────────────
def embed_texts(texts, tokenizer, model, prefix="", batch_size=32, max_length=128):
    all_embeddings = []
    texts = [f"{prefix}{str(t)}" for t in texts]

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        pooled = mean_pooling(output, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)  # L2 normalize
        all_embeddings.append(pooled.cpu().numpy())

        if (i // batch_size) % 20 == 0:
            print(f"    ... {min(i + batch_size, len(texts))}/{len(texts)} embedded")

    return np.vstack(all_embeddings)

# ── MODEL 1: QWEN3-EMBEDDING ─────────────────────────────────────────────────
print("\n=== Loading Qwen3-Embedding ===")
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", token=hf_token)
qwen_model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", token=hf_token).to(device)
qwen_model.eval()
print("Qwen3-Embedding loaded.")

for split_name, df in splits.items():
    print(f"\nEmbedding {split_name} set with Qwen3 ({len(df):,} rows)...")
    emb = embed_texts(df["text"].tolist(), qwen_tokenizer, qwen_model, prefix="")
    np.save(f"/kaggle/working/qwen3_{split_name}.npy", emb)
    print(f"  Saved qwen3_{split_name}.npy  shape={emb.shape}")

del qwen_model, qwen_tokenizer
torch.cuda.empty_cache()

# ── MODEL 2: E5-LARGE ─────────────────────────────────────────────────────────
print("\n=== Loading E5-Large ===")
e5_tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-large-v2", token=hf_token)
e5_model = AutoModel.from_pretrained("intfloat/e5-large-v2", token=hf_token).to(device)
e5_model.eval()
print("E5-Large loaded.")

# E5 REQUIRES a "query: " or "passage: " prefix for correct embeddings.
# We use "query: " uniformly here since we're embedding short standalone texts,
# not doing query-vs-passage retrieval.
for split_name, df in splits.items():
    print(f"\nEmbedding {split_name} set with E5-Large ({len(df):,} rows)...")
    emb = embed_texts(df["text"].tolist(), e5_tokenizer, e5_model, prefix="query: ")
    np.save(f"/kaggle/working/e5_{split_name}.npy", emb)
    print(f"  Saved e5_{split_name}.npy  shape={emb.shape}")

del e5_model, e5_tokenizer
torch.cuda.empty_cache()

# ── SAVE LABELS ALONGSIDE (for Phase 5) ──────────────────────────────────────
for split_name, df in splits.items():
    df[["mapped_label", "artifact_type"]].to_csv(
        f"/kaggle/working/{split_name}_labels.csv", index=False
    )

print("\n✅ Phase 4 complete. Embeddings + labels saved to /kaggle/working/")

HF token loaded.
Using device: cuda

=== Loading Qwen3-Embedding ===


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E5-Large loaded.

Embedding train set with E5-Large (59,105 rows)...
    ... 32/59105 embedded
    ... 672/59105 embedded
    ... 1312/59105 embedded
    ... 1952/59105 embedded
    ... 2592/59105 embedded
    ... 3232/59105 embedded
    ... 3872/59105 embedded
    ... 4512/59105 embedded
    ... 5152/59105 embedded
    ... 5792/59105 embedded
    ... 6432/59105 embedded
    ... 7072/59105 embedded
    ... 7712/59105 embedded
    ... 8352/59105 embedded
    ... 8992/59105 embedded
    ... 9632/59105 embedded
    ... 10272/59105 embedded
    ... 10912/59105 embedded
    ... 11552/59105 embedded
    ... 12192/59105 embedded
    ... 12832/59105 embedded
    ... 13472/59105 embedded
    ... 14112/59105 embedded
    ... 14752/59105 embedded
    ... 15392/59105 embedded
    ... 16032/59105 embedded
    ... 16672/59105 embedded
    ... 17312/59105 embedded
    ... 17952/59105 embedded
    ... 18592/59105 embedded
    ... 19232/59105 embedded
    ... 19872/59105 embedded
    ... 20512/59105 em

In [ ]:
# -----------------------------------------------------------------------------
# PHASE 5 — TRAIN THE CLASSIFIERS
# -----------------------------------------------------------------------------

In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
import joblib
import os

os.makedirs("/kaggle/working/models", exist_ok=True)

# ── LOAD EMBEDDINGS + LABELS ──────────────────────────────────────────────────
embedding_sets = {
    "qwen3": {
        "train": np.load("/kaggle/working/qwen3_train.npy"),
        "val":   np.load("/kaggle/working/qwen3_val.npy"),
        "test":  np.load("/kaggle/working/qwen3_test.npy"),
    },
    "e5": {
        "train": np.load("/kaggle/working/e5_train.npy"),
        "val":   np.load("/kaggle/working/e5_val.npy"),
        "test":  np.load("/kaggle/working/e5_test.npy"),
    },
}

labels = {
    "train": pd.read_csv("/kaggle/working/train_labels.csv"),
    "val":   pd.read_csv("/kaggle/working/val_labels.csv"),
    "test":  pd.read_csv("/kaggle/working/test_labels.csv"),
}

# add binary identification label
for split in labels:
    labels[split]["id_label"] = labels[split]["mapped_label"].apply(
        lambda x: "Not-SATD" if x == "Not-SATD" else "SATD"
    )

artifact_types = labels["train"]["artifact_type"].unique()
all_results = []

# ── HELPER: TRAIN + EVALUATE ONE MODEL ────────────────────────────────────────
def train_and_eval(X_train, y_train, X_val, y_val, X_test, y_test,
                    clf_name, embed_name, artifact_type, task_name):
    if clf_name == "xgboost":
        # encode labels as integers for xgboost
        classes = sorted(y_train.unique())
        class_to_idx = {c: i for i, c in enumerate(classes)}
        idx_to_class = {i: c for c, i in class_to_idx.items()}

        y_train_enc = y_train.map(class_to_idx)
        y_test_enc  = y_test.map(class_to_idx)

        clf = XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            eval_metric="mlogloss", random_state=42, n_jobs=-1
        )
        clf.fit(X_train, y_train_enc)
        y_pred_enc = clf.predict(X_test)
        y_pred = pd.Series(y_pred_enc).map(idx_to_class)

    else:  # logistic regression
        clf = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"  [{task_name}] {embed_name} + {clf_name} | {artifact_type} → macro F1: {macro_f1:.4f}")

    model_path = f"/kaggle/working/models/{task_name}_{artifact_type}_{embed_name}_{clf_name}.joblib"
    joblib.dump(clf, model_path)

    all_results.append({
        "task": task_name, "artifact_type": artifact_type,
        "embedding": embed_name, "classifier": clf_name,
        "macro_f1": macro_f1, "report": report
    })

# ── TASK 1: IDENTIFICATION (SATD vs Not-SATD) ─────────────────────────────────
print("=" * 70)
print("TASK 1: IDENTIFICATION (SATD vs Not-SATD)")
print("=" * 70)

for artifact_type in artifact_types:
    print(f"\n── {artifact_type} ──")
    train_mask = labels["train"]["artifact_type"] == artifact_type
    val_mask   = labels["val"]["artifact_type"] == artifact_type
    test_mask  = labels["test"]["artifact_type"] == artifact_type

    y_train = labels["train"].loc[train_mask, "id_label"]
    y_val   = labels["val"].loc[val_mask, "id_label"]
    y_test  = labels["test"].loc[test_mask, "id_label"]

    for embed_name, emb_data in embedding_sets.items():
        X_train = emb_data["train"][train_mask.values]
        X_val   = emb_data["val"][val_mask.values]
        X_test  = emb_data["test"][test_mask.values]

        for clf_name in ["xgboost", "logreg"]:
            train_and_eval(X_train, y_train, X_val, y_val, X_test, y_test,
                            clf_name, embed_name, artifact_type, "identification")

# ── TASK 2: CATEGORIZATION (C/D, REQ, TES, DOC) — SATD-only rows ─────────────
print("\n" + "=" * 70)
print("TASK 2: CATEGORIZATION (C/D vs REQ vs TES vs DOC)")
print("=" * 70)

for artifact_type in artifact_types:
    print(f"\n── {artifact_type} ──")
    train_mask = (labels["train"]["artifact_type"] == artifact_type) & (labels["train"]["id_label"] == "SATD")
    val_mask   = (labels["val"]["artifact_type"] == artifact_type)   & (labels["val"]["id_label"] == "SATD")
    test_mask  = (labels["test"]["artifact_type"] == artifact_type)  & (labels["test"]["id_label"] == "SATD")

    y_train = labels["train"].loc[train_mask, "mapped_label"]
    y_val   = labels["val"].loc[val_mask, "mapped_label"]
    y_test  = labels["test"].loc[test_mask, "mapped_label"]

    for embed_name, emb_data in embedding_sets.items():
        X_train = emb_data["train"][train_mask.values]
        X_val   = emb_data["val"][val_mask.values]
        X_test  = emb_data["test"][test_mask.values]

        for clf_name in ["xgboost", "logreg"]:
            train_and_eval(X_train, y_train, X_val, y_val, X_test, y_test,
                            clf_name, embed_name, artifact_type, "categorization")

# ── SAVE RESULTS SUMMARY ───────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)[["task", "artifact_type", "embedding", "classifier", "macro_f1"]]
results_df.to_csv("/kaggle/working/phase5_results_summary.csv", index=False)

print("\n\n --------- Phase 5 complete.")
print("\nResults summary:")
print(results_df.to_string(index=False))

TASK 1: IDENTIFICATION (SATD vs Not-SATD)

── code_comment ──
  [identification] qwen3 + xgboost | code_comment → macro F1: 0.9152
  [identification] qwen3 + logreg | code_comment → macro F1: 0.9079
  [identification] e5 + xgboost | code_comment → macro F1: 0.8911
  [identification] e5 + logreg | code_comment → macro F1: 0.8708

── commit ──
  [identification] qwen3 + xgboost | commit → macro F1: 0.6861
  [identification] qwen3 + logreg | commit → macro F1: 0.5822
  [identification] e5 + xgboost | commit → macro F1: 0.7508
  [identification] e5 + logreg | commit → macro F1: 0.6050

── issue ──
  [identification] qwen3 + xgboost | issue → macro F1: 0.7981
  [identification] qwen3 + logreg | issue → macro F1: 0.7858
  [identification] e5 + xgboost | issue → macro F1: 0.7779
  [identification] e5 + logreg | issue → macro F1: 0.7656

── pull_request ──
  [identification] qwen3 + xgboost | pull_request → macro F1: 0.7924
  [identification] qwen3 + logreg | pull_request → macro F1: 0.7476
  

In [ ]:
# phase 6

In [13]:
import pandas as pd

# ── BASELINE NUMBERS (Paper + Supervisor's replication for CM) ───────────────
baseline_identification = {
    "code_comment":  {"Not-SATD": 0.952, "SATD": 0.927, "macro_f1": 0.939, "source": "Paper (BiLSTM+AugGPT)"},
    "issue":         {"Not-SATD": 0.937, "SATD": 0.820, "macro_f1": 0.878, "source": "Paper (BiLSTM+AugGPT)"},
    "pull_request":  {"Not-SATD": 0.917, "SATD": 0.806, "macro_f1": 0.862, "source": "Paper (BiLSTM+AugGPT)"},
    "commit":        {"Not-SATD": None,  "SATD": None,  "macro_f1": 0.91,  "source": "Supervisor's replication (BiLSTM)"},
}

baseline_categorization = {
    "code_comment":  {"C/D": 0.885, "DOC": 0.925, "TES": 0.925, "REQ": 0.796, "macro_f1": 0.882, "source": "Paper (BERT+AugGPT)"},
    "issue":         {"C/D": 0.902, "DOC": 0.922, "TES": 0.922, "REQ": 0.851, "macro_f1": 0.899, "source": "Paper (BERT+AugGPT)"},
    "pull_request":  {"C/D": 0.842, "DOC": 0.895, "TES": 0.851, "REQ": 0.842, "macro_f1": 0.876, "source": "Paper (BERT+AugGPT)"},
    "commit":        {"C/D": None,  "DOC": None,  "TES": None,  "REQ": None,  "macro_f1": 0.9804, "source": "Supervisor's replication (BERT)"},
}

# ── LOAD PHASE 5 RESULTS ───────────────────────────────────────────────────────
results_df = pd.read_csv("/kaggle/working/phase5_results_summary.csv")

# ── BUILD COMPARISON TABLES ───────────────────────────────────────────────────
def build_comparison(task_name, baseline_dict):
    task_results = results_df[results_df["task"] == task_name].copy()

    rows = []
    for artifact_type in baseline_dict:
        baseline_f1 = baseline_dict[artifact_type]["macro_f1"]
        baseline_source = baseline_dict[artifact_type]["source"]

        rows.append({
            "artifact_type": artifact_type,
            "embedding": "—",
            "classifier": baseline_source,
            "macro_f1": baseline_f1,
            "vs_baseline": "—  (baseline)"
        })

        artifact_results = task_results[task_results["artifact_type"] == artifact_type]
        for _, r in artifact_results.iterrows():
            diff = r["macro_f1"] - baseline_f1
            sign = "+" if diff >= 0 else ""
            rows.append({
                "artifact_type": artifact_type,
                "embedding": r["embedding"],
                "classifier": r["classifier"],
                "macro_f1": round(r["macro_f1"], 4),
                "vs_baseline": f"{sign}{diff:.4f}"
            })

    return pd.DataFrame(rows)

identification_comparison = build_comparison("identification", baseline_identification)
categorization_comparison = build_comparison("categorization", baseline_categorization)

print("=" * 90)
print("IDENTIFICATION TASK — Macro F1 vs Paper / Supervisor Baseline")
print("=" * 90)
print(identification_comparison.to_string(index=False))

print("\n" + "=" * 90)
print("CATEGORIZATION TASK — Macro F1 vs Paper / Supervisor Baseline")
print("=" * 90)
print(categorization_comparison.to_string(index=False))

# ── SAVE FINAL OUTPUT ──────────────────────────────────────────────────────────
identification_comparison.to_csv("/kaggle/working/phase6_identification_comparison.csv", index=False)
categorization_comparison.to_csv("/kaggle/working/phase6_categorization_comparison.csv", index=False)

# ── BEST EMBEDDING PER TASK (the supervisor's actual question) ───────────────
print("\n" + "=" * 90)
print("BEST EMBEDDING MODEL — averaged across all artifact types")
print("=" * 90)

for task_name in ["identification", "categorization"]:
    task_results = results_df[results_df["task"] == task_name]
    avg_by_embedding = task_results.groupby("embedding")["macro_f1"].mean().sort_values(ascending=False)
    print(f"\n{task_name.upper()}:")
    print(avg_by_embedding.round(4))

avg_by_combo = results_df.groupby(["embedding", "classifier"])["macro_f1"].mean().sort_values(ascending=False)
print("\nOVERALL BEST EMBEDDING + CLASSIFIER COMBO (avg across both tasks, all artifacts):")
print(avg_by_combo.round(4))

print("\n✅ Phase 6 complete. Saved phase6_identification_comparison.csv and phase6_categorization_comparison.csv")

IDENTIFICATION TASK — Macro F1 vs Paper / Supervisor Baseline
artifact_type embedding                        classifier  macro_f1   vs_baseline
 code_comment         —             Paper (BiLSTM+AugGPT)    0.9390 —  (baseline)
 code_comment     qwen3                           xgboost    0.9152       -0.0238
 code_comment     qwen3                            logreg    0.9079       -0.0311
 code_comment        e5                           xgboost    0.8911       -0.0479
 code_comment        e5                            logreg    0.8708       -0.0682
        issue         —             Paper (BiLSTM+AugGPT)    0.8780 —  (baseline)
        issue     qwen3                           xgboost    0.7981       -0.0799
        issue     qwen3                            logreg    0.7858       -0.0922
        issue        e5                           xgboost    0.7779       -0.1001
        issue        e5                            logreg    0.7656       -0.1124
 pull_request         —             

In [14]:
# =========================================================================================
# phase 7
# =========================================================================================